In [0]:
%run "./000-config"

### Data generation:

For this project, we will use demo data. To generate the demo data we will use data generator. The datasets include:
- billing_plans: Telcos provide multiple billing plans to their consumers. These are representative plans.
- customers: Telco customer master data including id, name, phone number, email etc.
- billing_items: The detailed billing items including device_id, event type (data, call or text), call duration, data transferred, event timestamp etc.
- invoice: Telco billing invoice for the customer

The invoice is based on:
- Cost per MB of internet activity
- Cost per minute of call for each of the call categories
- cost per message

### Databricks Labs Data Generator

In [0]:
%pip install dbldatagen

In [0]:
# Create a catalog and a database (schema) to store the delta tables for our data
catalog = config['catalog']
db = config['database']

print(catalog, db)

In [0]:
# Ensure the catalog existss, create if it does not exists
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# Ensure the schema exists within the specified catalog, create it if it does not
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")


In [0]:
# Configurations for data generation
UNIQUE_CUSTOMERS = config['UNIQUE_CUSTOMERS']
CUSTOMER_MIN_VALUE = config['CUSTOMER_MIN_VALUE']
DEVICE_MIN_VALUE = config['DEVICE_MIN_VALUE']
SUBSCRIBER_NUM_MIN_VALUE = config['SUBSCRIBER_NUM_MIN_VALUE']
UNIQUE_PLANS = config['UNIQUE_PLANS']
PLAN_MIN_VALUE = config['PLAN_MIN_VALUE']

AVG_EVENTS_PER_CUSTOMER = config['AVG_EVENTS_PER_CUSTOMER']

shuffle_partitions_requested = config['shuffle_partitions_requested']
partitions_requested = config['partitions_requested']
NUM_DAYS = config['NUM_DAYS']
MB_100 = config['MB_100']
K_1 = config['K_1']
start_dt = config['start_dt']
end_dt = config['end_dt']
vector_search_index = config['catalog']

llm_endpoint = config['llm_endpoint']
warehouse_id = config['warehouse_id']

In [0]:
# Create billing plans table
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DecimalType

# Define the schema for the JSON data
schema = StructType([
    StructField("Plan_key", IntegerType(), True),
    StructField("Plan_id", StringType(), True),
    StructField("Plan_name", StringType(), True),
    StructField("contract_in_months", IntegerType(), True),
    StructField("monthly_charges_dollars", DoubleType(), True),
    StructField("Calls_Text", StringType(), True),
    StructField("Internet_Speed_MBPS", StringType(), True),
    StructField("Data_Limit_GB", StringType(), True),
    StructField("Data_Outside_Allowance_Per_MB", DoubleType(), True),
    StructField("Roam_Data_charges_per_MB", DoubleType(), True),
    StructField("Roam_Call_charges_per_min", DoubleType(), True),
    StructField("Roam_text_charges", DoubleType(), True),
    StructField("International_call_charge_per_min", DoubleType(), True),
    StructField("International_text_charge", DoubleType(), True)
])

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

ws = WorkspaceClient(profile="DEV")

user = ws.current_user.me().user_name
folder = f"/Users/{user}/telco-billing-customer-care/data"
if not folder:
    ws.workspace.mkdirs(folder)

In [0]:
import os

In [0]:
def workspace_uri(*parts: str) -> str:
    """Build a file:/Workspace/... path, even if a part starts with '/' or 'Workspace'."""
    rel = "/".join(p.strip("/") for p in parts)
    if not rel.startswith("Workspace/"):
        rel = f"Workspace/{rel}"
    return f"file:/{rel}"


plans_path = workspace_uri(folder, "billing_plans.json")
print(plans_path)  # should look like file:/Workspace/Users/<email>/.../billing_plans.json

df_plans = spark.read.format("json").schema(schema).load(plans_path)
display(df_plans)

### Generate customer, billing items and invoice datasets

In [0]:
%pip install jmespath pyparsing

In [0]:
import dbldatagen as dg
import pyspark.sql.functions as F

shuffle_partitions_requested = 8
partitions_requested = 1

spark.conf.set("spark.sql.shuffle.partitions", shuffle_partitions_requested)
#spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
#spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 20000)


#spark.catalog.clearCache()  # clear cache so that if we run multiple times to check performance, we're not relying on cache
shuffle_partitions_requested = 8
partitions_requested = 8
data_rows = UNIQUE_CUSTOMERS


In [0]:
customer_dataspec = (dg.DataGenerator(spark, rows=data_rows, partitions=partitions_requested)
            .withColumn("customer_id","decimal(10)", minValue=CUSTOMER_MIN_VALUE, uniqueValues=UNIQUE_CUSTOMERS)
            .withColumn("customer_name", template=r"\\w \\w|\\w a. \\w")  
           
            # use the following for a simple sequence
            #.withColumn("device_id","decimal(10)", minValue=DEVICE_MIN_VALUE, uniqueValues=UNIQUE_CUSTOMERS)
                     
            .withColumn("device_id","decimal(10)",  minValue=DEVICE_MIN_VALUE, 
                        baseColumn="customer_id", baseColumnType="hash")

            .withColumn("phone_number","decimal(10)",  minValue=SUBSCRIBER_NUM_MIN_VALUE, 
                        baseColumn=["customer_id", "customer_name"], baseColumnType="hash")

            # for email, we'll just use the formatted phone number
            .withColumn("email","string",  format="subscriber_%s@myoperator.com", baseColumn="phone_number")
            .withColumn("plan", "int", minValue=PLAN_MIN_VALUE, uniqueValues=UNIQUE_PLANS, random=True)
            .withColumn("contract_start_dt", "date", data_range=dg.DateRange("2023-01-01 00:00:00",
                                                                             "2024-12-31 11:59:59",
                                                                             "days=1"),
                        random=True)
            )

In [0]:
df_customers = (customer_dataspec.build()
                .dropDuplicates(["device_id"])
                .dropDuplicates(["phone_number"])
                .orderBy("customer_id")
                .cache()
               )

In [0]:
display(df_customers)

In [0]:
# Write the DataFrame to a Delta table
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{db}.customers")
